# System Design Agent

An autonomous agent that:
- **Searches the web** (DuckDuckGo via MCP) for latest technologies and patterns
- **Reads / writes files** in `system-design/` (filesystem MCP server)
- **Queries the RAG system** for existing system designs stored in Oracle DB
- **Ingests new designs** into Oracle DB when they are not already present
- **Maintains a knowledge graph** of system design components in Oracle DB
- Produces a structured Markdown design document per requirement

**Prerequisites:**
- `ollama serve` with `llama3.2` and `nomic-embed-text` pulled
- Oracle DB env vars: `ORACLE_USER`, `ORACLE_PASS`, `ORACLE_TLS_DNS`
- Node.js available (for `npx duckduckgo-mcp-server`)
- `fastmcp` installed (`pip install fastmcp`)

## 1. Install dependencies (run once)

In [ ]:
# !pip install fastmcp langchain-ollama langchain-oracledb langchain-core langgraph oracledb pydantic

## 2. Oracle DB connection

In [ ]:
import os
import oracledb

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASS"],
    dsn=os.environ["ORACLE_TLS_DNS"]
)
print("Connected to Oracle DB")

## 3. Embedding model + Vector stores

Two dedicated tables:
- `SYSTEM_DESIGN_VECTORS` — chunked design documents for RAG
- `SYSTEM_DESIGN_GRAPH` — knowledge graph nodes (component, type, description, relationships)

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_oracledb.vectorstores import DistanceStrategy

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

# RAG store — chunked system design documents
rag_store = OracleVS(
    client=connection,
    table_name="SYSTEM_DESIGN_VECTORS",
    embedding_function=embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

# Knowledge graph store — component nodes
kg_store = OracleVS(
    client=connection,
    table_name="SYSTEM_DESIGN_GRAPH",
    embedding_function=embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

print("Vector stores ready: SYSTEM_DESIGN_VECTORS, SYSTEM_DESIGN_GRAPH")

## 4. LLM

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0)

## 5. RAG helpers (retrieval + ingestion)

In [ ]:
from langchain_core.documents import Document


def rag_retrieve(query: str, k: int = 4) -> str:
    """Retrieve top-k relevant chunks from SYSTEM_DESIGN_VECTORS."""
    results = rag_store.similarity_search(query, k=k)
    if not results:
        return "No existing design found for this query."
    parts = []
    for i, doc in enumerate(results, 1):
        src = doc.metadata.get("source", "unknown")
        parts.append(f"[{i}] (source: {src})\n{doc.page_content.strip()}")
    return "\n\n".join(parts)


def rag_ingest(text: str, source: str) -> str:
    """Chunk and ingest a design document into SYSTEM_DESIGN_VECTORS."""
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)
    docs = [Document(page_content=c, metadata={"source": source}) for c in chunks]
    rag_store.add_documents(docs)
    return f"Ingested {len(docs)} chunks from '{source}' into SYSTEM_DESIGN_VECTORS"


def kg_add_component(name: str, component_type: str, description: str, relationships: str = "") -> str:
    """Add a system design component node to SYSTEM_DESIGN_GRAPH."""
    content = f"Component: {name}\nType: {component_type}\nDescription: {description}\nRelationships: {relationships}"
    doc = Document(
        page_content=content,
        metadata={"name": name, "type": component_type, "relationships": relationships}
    )
    kg_store.add_documents([doc])
    return f"Knowledge graph: added component '{name}' ({component_type})"


def kg_query(query: str, k: int = 5) -> str:
    """Search the knowledge graph for related components."""
    results = kg_store.similarity_search(query, k=k)
    if not results:
        return "No components found in the knowledge graph."
    parts = []
    for doc in results:
        parts.append(doc.page_content.strip())
    return "\n---\n".join(parts)


print("RAG helpers defined")

## 6. MCP tool loader (mcp SDK >= 2.0, no langchain-mcp-adapters)

In [ ]:
from typing import Any, Optional, Literal
from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters
from langchain_core.tools import StructuredTool
from pydantic import create_model


def mcp_tool_to_langchain(mcp_tool, session: ClientSession) -> StructuredTool:
    """Wrap an MCP tool as a LangChain StructuredTool (mcp SDK >= 2.0)."""
    properties = mcp_tool.input_schema.get("properties", {})
    required = set(mcp_tool.input_schema.get("required", []))

    fields = {}
    for name, schema in properties.items():
        t = schema.get("type")
        enum_vals = schema.get("enum")
        default = schema.get("default")
        is_required = name in required

        if enum_vals:
            annotation = Literal[tuple(enum_vals)]
        elif t == "integer":
            annotation = int
        elif t == "number":
            annotation = float
        elif t == "boolean":
            annotation = bool
        else:
            annotation = str

        if is_required:
            fields[name] = (annotation, ...)
        elif default is not None:
            fields[name] = (Optional[annotation], default)
        else:
            fields[name] = (Optional[annotation], None)

    ArgsModel = create_model(f"{mcp_tool.name}_args", **fields)

    async def call_tool(**kwargs: Any) -> str:
        clean_args = {k: v for k, v in kwargs.items() if v is not None}
        result = await session.call_tool(mcp_tool.name, arguments=clean_args)
        return "\n".join(
            block.text for block in result.content if hasattr(block, "text")
        )

    return StructuredTool.from_function(
        coroutine=call_tool,
        name=mcp_tool.name,
        description=mcp_tool.description or "",
        args_schema=ArgsModel,
    )


print("MCP tool loader defined")

## 7. Python tools (RAG + knowledge graph wrapped as LangChain tools)

In [ ]:
from langchain_core.tools import tool


@tool
def search_existing_designs(query: str) -> str:
    """Search SYSTEM_DESIGN_VECTORS in Oracle DB for existing system designs related to the query. Use this before generating a new design."""
    return rag_retrieve(query)


@tool
def ingest_design_to_rag(text: str, source: str) -> str:
    """Ingest a system design document (as text) into SYSTEM_DESIGN_VECTORS in Oracle DB. Use source as a short identifier e.g. 'inventory-search-v1'."""
    return rag_ingest(text, source)


@tool
def add_component_to_knowledge_graph(name: str, component_type: str, description: str, relationships: str = "") -> str:
    """Add a system component node to the knowledge graph in SYSTEM_DESIGN_GRAPH Oracle table. component_type examples: API Gateway, Message Queue, Vector DB, Cache, Service. relationships is a comma-separated list of component names this connects to."""
    return kg_add_component(name, component_type, description, relationships)


@tool
def query_knowledge_graph(query: str) -> str:
    """Search the knowledge graph (SYSTEM_DESIGN_GRAPH) for existing components related to the query."""
    return kg_query(query)


python_tools = [
    search_existing_designs,
    ingest_design_to_rag,
    add_component_to_knowledge_graph,
    query_knowledge_graph,
]

print(f"Python tools defined: {[t.name for t in python_tools]}")

## 8. Build the agent (MCP tools + Python tools)

In [ ]:
from mcp.client.stdio import stdio_client
from langgraph.prebuilt import create_react_agent

SYSTEM_DESIGN_DIR = "/Users/zainabfirdaus/git/learn/system-design"
MCP_SERVER_PATH = "/Users/zainabfirdaus/git/learn/python3/mcps/mcp_filesystem_server.py"

SYSTEM_PROMPT = f"""\
You are an expert System Design Architect.

You have access to the following tools:

FILESYSTEM (via MCP):
  - read_file: read any file
  - list_files: list files in a directory
  - write_file: write or overwrite a file (creates parent dirs automatically)
  - create_directory: create a new folder

WEB SEARCH (via MCP):
  - duckduckgo_web_search: search the web for latest technologies and best practices

RAG / KNOWLEDGE GRAPH (Python tools):
  - search_existing_designs: search Oracle DB for existing system designs
  - ingest_design_to_rag: ingest a new design document into Oracle DB
  - add_component_to_knowledge_graph: add a component node to the knowledge graph
  - query_knowledge_graph: search the knowledge graph for existing components

WORKFLOW for each requirement:
1. Search existing designs using search_existing_designs.
2. Query the knowledge graph using query_knowledge_graph.
3. If no relevant design exists, search the web for best practices.
4. Create a new folder under: {SYSTEM_DESIGN_DIR}/<requirement-slug>/
5. Write the design as a Markdown file: design.md with sections:
   - Overview, Architecture Diagram (text-based), Components,
     Data Flow, Scalability, Edge Cases, Technology Choices.
6. Ingest the written design into Oracle DB using ingest_design_to_rag.
7. Add each major component to the knowledge graph using add_component_to_knowledge_graph.

Always write concrete, production-ready designs. Cover scalability and edge cases.
Base file paths on: {SYSTEM_DESIGN_DIR}
"""

# MCP server params
fs_server_params = StdioServerParameters(
    command="python",
    args=[MCP_SERVER_PATH],
)

ddg_server_params = StdioServerParameters(
    command="npx",
    args=["-y", "duckduckgo-mcp-server"],
)

print("Agent config ready")

## 9. Run — design a system

In [ ]:
async def run_system_design_agent(requirement: str) -> str:
    """Run the system design agent for a given requirement."""
    async with stdio_client(fs_server_params) as (fs_read, fs_write):
        async with ClientSession(fs_read, fs_write) as fs_session:
            await fs_session.initialize()
            fs_response = await fs_session.list_tools()
            fs_tools = [mcp_tool_to_langchain(t, fs_session) for t in fs_response.tools]

            async with stdio_client(ddg_server_params) as (ddg_read, ddg_write):
                async with ClientSession(ddg_read, ddg_write) as ddg_session:
                    await ddg_session.initialize()
                    ddg_response = await ddg_session.list_tools()
                    ddg_tools = [mcp_tool_to_langchain(t, ddg_session) for t in ddg_response.tools]

                    all_tools = fs_tools + ddg_tools + python_tools
                    print(f"All tools: {[t.name for t in all_tools]}")

                    agent = create_react_agent(
                        model=llm,
                        tools=all_tools,
                        prompt=SYSTEM_PROMPT
                    )

                    result = await agent.ainvoke({
                        "messages": [{"role": "user", "content": requirement}]
                    })

                    return result["messages"][-1].content


# ── Run ────────────────────────────────────────────────────────────────────
requirement = """
Design a system which can enable users to find a product in the inventory,
search for the product using natural language,
and fetch a particular model based on user query.
"""

output = await run_system_design_agent(requirement)
print(output)

## 10. Interactive — provide your own requirement

In [ ]:
custom_requirement = input("Enter your system design requirement: ").strip()
output = await run_system_design_agent(custom_requirement)
print(output)

## 11. Inspect what's been stored

In [ ]:
# Browse existing designs in RAG store
query = "inventory search natural language"
print("=" * 60)
print(f"RAG search: '{query}'")
print("=" * 60)
print(rag_retrieve(query, k=3))

print()

# Browse knowledge graph
print("=" * 60)
print("Knowledge graph: search 'vector search API'")
print("=" * 60)
print(kg_query("vector search API", k=5))